In [6]:
# Fold 0 grouped-model timing run: baseline hyperparameters
from pathlib import Path
import copy
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

repo = Path.cwd()
while not (repo / "data/ready_full/fold0").is_dir():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from model.grouped_cnn_gru_attention import FullSensorGroupedCNNGRUAttention
from utils.evaluation import competition_score

FOLD = 0
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
MIN_LEARNING_RATE = 1e-5
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0  # Simple and reliable for Windows notebooks.

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; this timing run is intended for your GPU.")
device = torch.device("cuda")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Fold {FOLD}, seed {SEED}, device: {torch.cuda.get_device_name(0)}")


Fold 0, seed 42, device: NVIDIA GeForce RTX 3050 Laptop GPU


In [7]:
# Load only the selected fold. Memory mapping avoids copying the ~2 GB training array into RAM.
class GestureDataset(Dataset):
    def __init__(self, fold_dir, split):
        self.x = np.load(fold_dir / f"X_{split}.npy", mmap_mode="r")
        self.mask = np.load(fold_dir / f"mask_{split}.npy", mmap_mode="r")
        self.y = np.load(fold_dir / f"y_gesture_{split}.npy", mmap_mode="r")
        if not (len(self.x) == len(self.mask) == len(self.y)):
            raise ValueError(f"Mismatched {split} array lengths")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        # Make writable per-sample copies before converting read-only memmap slices to tensors.
        x = torch.from_numpy(np.array(self.x[index], dtype=np.float32, copy=True))
        mask = torch.from_numpy(np.array(self.mask[index], copy=True))
        y = torch.tensor(int(self.y[index]), dtype=torch.long)
        return x, mask, y

fold_dir = repo / "data/ready_full" / f"fold{FOLD}"
train_dataset = GestureDataset(fold_dir, "train")
val_dataset = GestureDataset(fold_dir, "val")
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_dataset)} sequences; validation: {len(val_dataset)} sequences")


Train: 6427 sequences; validation: 1627 sequences


In [8]:
# Grouped CNN-GRU-attention model with the same 18-class output as the baseline.
model = FullSensorGroupedCNNGRUAttention(num_classes=18).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Trainable parameters: 1,591,728


In [4]:
from utils.train import train_one_epoch, validate

# Run ONE fold. Stop when the competition score has not improved for PATIENCE epochs.
history = []
best_score = float("-inf")
best_epoch = 0
best_metrics = None
best_weights = None
epochs_without_improvement = 0

torch.cuda.synchronize()
run_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP_NORM)
    val_loss, metrics, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step()
    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **metrics,
        "seconds": epoch_seconds,
    })
    print(
        f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
        f"| binary F1 {metrics['binary_f1']:.4f} | macro F1 {metrics['macro_f1']:.4f} "
        f"| score {metrics['score']:.4f} | {epoch_seconds:.1f}s"
    )

    if metrics["score"] > best_score:
        best_score = metrics["score"]
        best_epoch = epoch
        best_metrics = metrics.copy()
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

fold_seconds = time.perf_counter() - run_start
model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch}")
print(f"Best binary F1: {best_metrics['binary_f1']:.4f}")
print(f"Best macro F1: {best_metrics['macro_f1']:.4f}")
print(f"Best validation score: {best_score:.4f}")
print(f"Fold {FOLD} training + validation time: {fold_seconds / 60:.1f} minutes")
print(f"Rough five-fold estimate: {5 * fold_seconds / 60:.1f} minutes")
print(f"Difference from existing fold-0 ToF score (0.8273): {best_score - 0.8273:+.4f}")


Epoch 01 | train loss 1.4771 | val loss 1.0968 | binary F1 0.9768 | macro F1 0.5245 | score 0.7506 | 63.4s
Epoch 02 | train loss 0.9222 | val loss 0.8914 | binary F1 0.9717 | macro F1 0.6029 | score 0.7873 | 47.9s
Epoch 03 | train loss 0.7720 | val loss 0.8212 | binary F1 0.9727 | macro F1 0.6426 | score 0.8077 | 76.9s
Epoch 04 | train loss 0.6913 | val loss 0.7944 | binary F1 0.9892 | macro F1 0.6605 | score 0.8248 | 44.3s
Epoch 05 | train loss 0.6209 | val loss 0.7721 | binary F1 0.9912 | macro F1 0.6629 | score 0.8270 | 42.9s
Epoch 06 | train loss 0.5534 | val loss 0.8187 | binary F1 0.9867 | macro F1 0.6479 | score 0.8173 | 42.8s
Epoch 07 | train loss 0.5233 | val loss 0.8070 | binary F1 0.9808 | macro F1 0.6786 | score 0.8297 | 42.8s
Epoch 08 | train loss 0.4816 | val loss 0.8080 | binary F1 0.9857 | macro F1 0.6876 | score 0.8366 | 54.9s
Epoch 09 | train loss 0.4399 | val loss 0.8343 | binary F1 0.9852 | macro F1 0.6902 | score 0.8377 | 43.4s
Epoch 10 | train loss 0.4039 | val lo

In [9]:
# Train the grouped model independently on folds 0-4, using the pilot settings.
from tqdm.auto import tqdm
from utils.train import train_one_epoch, validate

checkpoint_dir = repo / "model"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
model_kwargs = {"num_classes": 18, "widths": (64, 96), "embedding_size": 64, "fusion_size": 192, "dropout": 0.2}

cv_results = []
cv_history = []
oof_true_parts, oof_pred_parts, oof_fold_parts = [], [], []

for fold in range(5):
    # Reset the seed so each fold starts reproducibly from a fresh model.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    fold_dir = repo / "data/ready_full" / f"fold{fold}"
    fold_train = GestureDataset(fold_dir, "train")
    fold_val = GestureDataset(fold_dir, "val")
    fold_train_loader = DataLoader(
        fold_train, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    fold_val_loader = DataLoader(
        fold_val, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    # Keep the exact feature order with the checkpoint for later distillation.
    with np.load(repo / f"data/ready_full/stats_fold{fold}.npz") as stats:
        feature_columns = [str(name) for name in stats["cols"]] + [
            str(name) for name in stats["flag_cols"]
        ]
    if len(feature_columns) != 346:
        raise ValueError(f"Fold {fold}: expected 346 input features")

    fold_model = FullSensorGroupedCNNGRUAttention(**model_kwargs).to(device)
    checkpoint_path = checkpoint_dir / f"grouped_cnn_gru_attention_fold{fold}_seed{SEED}.pt"
    fold_criterion = nn.CrossEntropyLoss()
    fold_optimizer = torch.optim.AdamW(
        fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    fold_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        fold_optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
    )

    best_score = float("-inf")
    best_epoch = 0
    best_metrics = None
    best_true = best_pred = None
    stale_epochs = 0
    torch.cuda.synchronize()
    fold_start = time.perf_counter()

    epoch_bar = tqdm(range(1, MAX_EPOCHS + 1), desc=f"Fold {fold}", unit="epoch")
    for epoch in epoch_bar:
        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(
            fold_model, fold_train_loader, fold_optimizer, fold_criterion,
            device, GRAD_CLIP_NORM,
        )
        val_loss, metrics, y_true, y_pred = validate(
            fold_model, fold_val_loader, fold_criterion, device,
        )
        fold_scheduler.step()
        torch.cuda.synchronize()
        epoch_seconds = time.perf_counter() - epoch_start

        cv_history.append({
            "fold": fold, "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_loss, **metrics, "seconds": epoch_seconds,
        })
        if metrics["score"] > best_score:
            best_score = metrics["score"]
            best_epoch = epoch
            best_metrics = metrics.copy()
            best_true, best_pred = y_true.copy(), y_pred.copy()
            stale_epochs = 0
            checkpoint = {
                "model_class": "model.grouped_cnn_gru_attention.FullSensorGroupedCNNGRUAttention",
                "model_kwargs": model_kwargs.copy(),
                "model_state_dict": {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in fold_model.state_dict().items()
                },
                "fold": fold,
                "seed": SEED,
                "best_epoch": epoch,
                "val_metrics": {name: float(value) for name, value in metrics.items()},
                "preprocessing_stats": f"data/ready_full/stats_fold{fold}.npz",
                "feature_columns": feature_columns,
                "training_config": {
                    "batch_size": BATCH_SIZE,
                    "max_epochs": MAX_EPOCHS,
                    "patience": PATIENCE,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "min_learning_rate": MIN_LEARNING_RATE,
                    "grad_clip_norm": GRAD_CLIP_NORM,
                },
            }
            temporary_path = checkpoint_path.with_suffix(".tmp")
            torch.save(checkpoint, temporary_path)
            temporary_path.replace(checkpoint_path)
        else:
            stale_epochs += 1

        epoch_bar.set_postfix(
            score=f"{metrics['score']:.4f}",
            best=f"{best_score:.4f}",
            seconds=f"{epoch_seconds:.0f}",
        )
        if stale_epochs >= PATIENCE:
            break

    torch.cuda.synchronize()
    fold_seconds = time.perf_counter() - fold_start
    cv_results.append({
        "fold": fold, "train_n": len(fold_train), "val_n": len(fold_val),
        "best_epoch": best_epoch, **best_metrics, "minutes": fold_seconds / 60,
        "checkpoint": str(checkpoint_path),
    })
    oof_true_parts.append(best_true)
    oof_pred_parts.append(best_pred)
    oof_fold_parts.append(np.full(len(best_true), fold, dtype=np.int8))
    print(
        f"Fold {fold}: best epoch {best_epoch}, "
        f"binary F1 {best_metrics['binary_f1']:.4f}, "
        f"macro F1 {best_metrics['macro_f1']:.4f}, "
        f"score {best_score:.4f}, time {fold_seconds / 60:.1f} min, "
        f"saved {checkpoint_path.name}"
    )

oof_true = np.concatenate(oof_true_parts)
oof_pred = np.concatenate(oof_pred_parts)
oof_fold = np.concatenate(oof_fold_parts)

# The mean across folds is the main CV comparison; pooled OOF F1 is also shown.
print("Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes")
print("-" * 78)
for row in cv_results:
    print(
        f"{row['fold']:>4} | {row['train_n']:>5} | {row['val_n']:>4} | "
        f"{row['best_epoch']:>10} | {row['binary_f1']:.4f}    | "
        f"{row['macro_f1']:.4f}   | {row['score']:.4f} | {row['minutes']:.1f}"
    )

for metric in ("binary_f1", "macro_f1", "score"):
    values = np.array([row[metric] for row in cv_results])
    print(f"{metric}: mean {values.mean():.4f}, SD {values.std(ddof=1):.4f}")

pooled_metrics = competition_score(oof_true, oof_pred)
print(
    f"Pooled out-of-fold ({len(oof_true)} sequences): "
    f"binary F1 {pooled_metrics['binary_f1']:.4f}, "
    f"macro F1 {pooled_metrics['macro_f1']:.4f}, "
    f"score {pooled_metrics['score']:.4f}"
)
print(f"Total five-fold time: {sum(row['minutes'] for row in cv_results):.1f} minutes")
print("Best-epoch checkpoints (one teacher per fold):")
for row in cv_results:
    print(f"  Fold {row['fold']}: {row['checkpoint']}")


Fold 0:  46%|████▌     | 23/50 [18:31<21:44, 48.32s/epoch, best=0.8536, score=0.8434, seconds=42]


Fold 0: best epoch 14, binary F1 0.9912, macro F1 0.7159, score 0.8536, time 18.5 min, saved grouped_cnn_gru_attention_fold0_seed42.pt


Fold 1:  50%|█████     | 25/50 [19:01<19:01, 45.66s/epoch, best=0.8361, score=0.8210, seconds=43]


Fold 1: best epoch 16, binary F1 0.9865, macro F1 0.6856, score 0.8361, time 19.0 min, saved grouped_cnn_gru_attention_fold1_seed42.pt


Fold 2: 100%|██████████| 50/50 [35:53<00:00, 43.08s/epoch, best=0.8695, score=0.8653, seconds=42]


Fold 2: best epoch 47, binary F1 0.9942, macro F1 0.7449, score 0.8695, time 35.9 min, saved grouped_cnn_gru_attention_fold2_seed42.pt


Fold 3:  68%|██████▊   | 34/50 [25:18<11:54, 44.65s/epoch, best=0.8744, score=0.8732, seconds=42]


Fold 3: best epoch 25, binary F1 0.9911, macro F1 0.7576, score 0.8744, time 25.3 min, saved grouped_cnn_gru_attention_fold3_seed42.pt


Fold 4:  52%|█████▏    | 26/50 [19:50<18:18, 45.79s/epoch, best=0.8429, score=0.8410, seconds=44]

Fold 4: best epoch 17, binary F1 0.9832, macro F1 0.7027, score 0.8429, time 19.8 min, saved grouped_cnn_gru_attention_fold4_seed42.pt
Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes
------------------------------------------------------------------------------
   0 |  6427 | 1627 |         14 | 0.9912    | 0.7159   | 0.8536 | 18.5
   1 |  6519 | 1535 |         16 | 0.9865    | 0.6856   | 0.8361 | 19.0
   2 |  6423 | 1631 |         47 | 0.9942    | 0.7449   | 0.8695 | 35.9
   3 |  6428 | 1626 |         25 | 0.9911    | 0.7576   | 0.8744 | 25.3
   4 |  6419 | 1635 |         17 | 0.9832    | 0.7027   | 0.8429 | 19.8
binary_f1: mean 0.9892, SD 0.0044
macro_f1: mean 0.7214, SD 0.0297
score: mean 0.8553, SD 0.0165
Pooled out-of-fold (8054 sequences): binary F1 0.9893, macro F1 0.7228, score 0.8560
Total five-fold time: 118.6 minutes
Best-epoch checkpoints (one teacher per fold):
  Fold 0: c:\Users\yuhen\Documents\GithubRepos\SC4000-Group-Assignment\model\grouped_c